# anatid as an agent's memory

Two integrations put the same file behind an agent. The OpenAI Agents SDK integration gives an
agent memory tools with an approval gate, keeps the conversation in the same file, and can park a
run that is waiting for a human. The MCP server exposes the verbs to any MCP client: Claude
Desktop, Claude Code, Cursor, or a script.

```
pip install "anatid[agents,mcp]"
```

This notebook runs offline: the agent uses the SDK's `ScriptedModel`, so the tool calls, the
interruption, the approval and the resume are real and only the model is not, and the MCP server
is driven by an in-process client.

In [1]:
import asyncio
import tempfile
from pathlib import Path

from anatid import Anatid, HashEmbedder

try:
    import agents  # noqa: F401
    HAVE_AGENTS = True
except ImportError:
    HAVE_AGENTS = False
try:
    import mcp.client  # noqa: F401
    HAVE_MCP = True
except ImportError:
    HAVE_MCP = False
print(f"openai-agents installed: {HAVE_AGENTS}; mcp installed: {HAVE_MCP}")

workdir = Path(tempfile.mkdtemp(prefix="anatid-notebook-"))

openai-agents installed: True; mcp installed: True


## 1. The OpenAI Agents SDK

`create_memory_tools(db)` returns nine function tools: `anatid_recall`, `anatid_context` and
`anatid_provenance` read freely; `anatid_remember`, `anatid_supersede`, `anatid_correct`,
`anatid_relate`, `anatid_unrelate` and `anatid_forget` are writes, and by default every write
interrupts the run until a person approves it (`approve_low_risk()` loosens that). `AnatidSession`
stores the conversation in the same file, and `RunStateStore` parks an interrupted run so it can be
resumed minutes or days later, in another process, with only the file and a run id.

In [2]:
if HAVE_AGENTS:
    from agents import Agent, Runner, set_tracing_disabled
    from agents.testing import ScriptedModel, assistant_message, function_call

    from anatid.integrations.openai_agents import AnatidSession, RunStateStore, create_memory_tools

    set_tracing_disabled(True)
    model = ScriptedModel([
        [function_call("anatid_remember",
                       {"content": "Ada prefers DuckDB for embedded analytics", "entities": ["Ada"], "kind": "preference"},
                       call_id="call-1")],
        [assistant_message("Saved that Ada prefers DuckDB.")],
    ])

    db = Anatid.open(workdir / "agent.anatid", tenant=1, embedding_dim=64, embedder=HashEmbedder(dim=64))
    session = AnatidSession("demo-conversation", db)
    store = RunStateStore(db)
    agent = Agent(name="assistant",
                  instructions="Remember durable facts the user tells you. Recall before answering.",
                  model=model, tools=create_memory_tools(db, session=session))

    result = await Runner.run(agent, "Remember that Ada prefers DuckDB.", session=session)
    await session.store_run_usage(result)
    print("interrupted on:", [i.tool_name for i in result.interruptions])
    print("memories written so far:", db.stats()["memories"])
else:
    print("pip install 'anatid[agents]' to run this section")

interrupted on: ['anatid_remember']
memories written so far: 0


The write did not happen: the run stopped before the tool body ran. Park it, approve it, resume it.

In [3]:
if HAVE_AGENTS:
    run_id = store.save_result(result, session_id=session.session_id)
    print("parked as run", run_id)

    # later, anywhere with the file:
    state = await store.resume(agent, run_id)
    for item in state.get_interruptions():
        print("approving", item.tool_name, getattr(item.raw_item, "arguments", ""))
        state.approve(item)
    store.mark_resolved(run_id, status="answered")
    result = await Runner.run(agent, state, session=session)

    print("agent:", result.final_output)
    print("memories written in this conversation:", [m.content for m in await session.memories_written_here()])
    print("entities mentioned:", await session.entities_mentioned())
    print("token usage:", await session.usage_totals())
    db.close()

parked as run run_e1ffbc04e2e34f60be587103d5c485ad
approving anatid_remember {"content":"Ada prefers DuckDB for embedded analytics","entities":["Ada"],"kind":"preference"}
agent: Saved that Ada prefers DuckDB.
memories written in this conversation: ['Ada prefers DuckDB for embedded analytics']
entities mentioned: [{'entity_id': 884810603352385536, 'name': 'Ada', 'kind': None, 'mentions': 4}]
token usage: {'requests': 1, 'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0, 'cached_tokens': 0, 'reasoning_tokens': 0, 'rows': 1}


## 2. The MCP server

`anatid-mcp` serves one database over the Model Context Protocol. The config block below is what
Claude Desktop, Claude Code and Cursor take; with `uv` there is nothing to locate, because
`uvx --with "anatid[mcp]" anatid --db ~/.anatid/memory.anatid` fetches and runs it.

```json
{"mcpServers": {"anatid": {"command": "uvx",
                            "args": ["--with", "anatid[mcp]", "anatid", "--db", "/Users/you/.anatid/memory.anatid"]}}}
```

Here the same server is built in process and driven with the MCP client library, which is how the
test suite exercises it.

In [4]:
if HAVE_MCP:
    from mcp.client import Client

    from anatid.integrations.mcp.server import ServerConfig, build_server

    mcp_db = Anatid.open(workdir / "shared.anatid", tenant=1, embedding_dim=64, embedder=HashEmbedder(dim=64))
    server = build_server(mcp_db, ServerConfig(db=str(mcp_db.path), tenant=1, env={}))

    async with Client(server) as client:
        tools = (await client.list_tools()).tools
        print("tools:", [t.name for t in tools])
        r = await client.call_tool("remember", {"content": "Ada leads Kestrel", "entities": ["Ada", "Kestrel"], "writer": "claude"})
        print("remember ->", r.structured_content["memory"]["content"], "id", r.structured_content["memory"]["memory_id"])
        r = await client.call_tool("relate", {"src": "Ada", "dst": "Kestrel", "rel_kind": "leads"})
        r = await client.call_tool("recall", {"query": "who leads Kestrel", "k": 3})
        print("recall  ->", [(h["content"], h["sources"]) for h in r.structured_content["hits"]],
              "weights", r.structured_content["weights"])
        r = await client.call_tool("stats", {})
        print("stats   ->", {k: r.structured_content["counts"][k] for k in ("memories", "entities", "edges_relates")})
else:
    print("pip install 'anatid[mcp]' to run this section")

tools: ['remember', 'relate', 'supersede', 'unrelate', 'correct', 'reinforce', 'forget', 'prune', 'rebuild_fts_index', 'recall', 'context', 'get', 'provenance', 'stats']
remember -> Ada leads Kestrel id 884810603922810880


recall  -> [('Ada leads Kestrel', ['vector', 'text', 'graph'])] weights {'vector': 1.0, 'text': 0.25, 'graph': 0.5}


stats   -> {'memories': 1, 'entities': 2, 'edges_relates': 1}


Every id crosses the wire as a decimal string, because JSON numbers lose precision above 2^53 and
an id that comes back rounded is useless as the argument to the next call.

With an extraction endpoint configured (`ANATID_EXTRACT_BASE_URL`, `ANATID_EXTRACT_MODEL`), or an
extractor handed to `build_server`, the server also offers `ingest` and `apply_patch`: the first
proposes a patch and returns its diff with a `patch_id`, the second commits it, and a patch is
applied at most once, so a client that retries after a lost reply gets the same receipt back.

In [5]:
if HAVE_MCP:
    from anatid.ingest import AddFact, MemoryPatch, Relation, ScriptedExtractor

    scripted = ScriptedExtractor([MemoryPatch(
        add_facts=(AddFact("Bo maintains the ingest service", ("Bo", "ingest service")),),
        add_relations=(Relation("Bo", "ingest service", "maintains"),),
    )])
    server = build_server(mcp_db, ServerConfig(db=str(mcp_db.path), tenant=1, env={}), extractor=scripted)
    async with Client(server) as client:
        proposed = (await client.call_tool("ingest", {"text": "Bo maintains the ingest service day to day.",
                                                      "source": "notes/2026-03-02.md"})).structured_content
        print("proposed patch", proposed["patch_id"], "\n   " + proposed["diff"].replace("\n", "\n   "))
        applied = (await client.call_tool("apply_patch", {"patch_id": proposed["patch_id"]})).structured_content
        print("applied:", applied["summary"])
        again = (await client.call_tool("apply_patch", {"patch_id": proposed["patch_id"]})).structured_content
        print("applied again? already_applied =", again["already_applied"])
    mcp_db.close()

proposed patch 884810604317075456 
   memory patch: 1 fact, 1 relation(s) added
   source: "Bo maintains the ingest service day to day." (43 chars)
     + fact        "Bo maintains the ingest service"  about: Bo, ingest service
     + relation    Bo -maintains-> ingest service
applied: episode 884810604329658368 stored; 1 memories created (884810604333852672); 1 relations opened.
applied again? already_applied = True


## 3. Sharing one memory

DuckDB gives one process exclusive use of a file, so two MCP servers cannot open the same database.
`anatid-server start --db memory.anatid --socket /tmp/anatid/anatid.sock` holds the file and serves
the verbs over a Unix socket; `anatid-mcp --socket ...` and `AnatidClient` talk to it, so several
agents share one memory with the same verbs and the same types. [`docs/server.md`](../../docs/server.md)
and [`docs/mcp.md`](../../docs/mcp.md) have the details, and
[`examples/server_demo.py`](../server_demo.py) runs the whole thing in twenty seconds.

In [6]:
import shutil

shutil.rmtree(workdir, ignore_errors=True)
print("cleaned up", workdir)

cleaned up /var/folders/1j/2k5xpt896fdb5rnbdm6nktnr0000gn/T/anatid-notebook-wyh_yjz2
